# SMDS morphology with Pandeia

Simulate a NIRCam image of a TLUSTY SMDS (or a placeholder SED) with a Sérsic profile, extract the azimuthal radial profile, and fit effective radius and Sérsic index by Monte Carlo plus Nelder–Mead.

Companion notebooks: `01_spectral_fitting.ipynb` (the SED), `03_he1640_absorption.ipynb`.

## Two original sources, one pipeline

- `SMDS_Morphology_Fitting_MC_mse.ipynb`: JADES candidates with a fixed best-fit TLUSTY file, exposure settings, and hand-entered F200W error bars. Loss is the mean squared error of peak-normalized profiles. 75 Monte Carlo draws. Sérsic index starts near 4 for the JADES-GS-z13 run.
- `pandeia_radial_profile_fitting_notebook`: the same fit for the 15 Ferrara, Pallottini & Sommovigo (2024) “blue monsters”, with a mock spectrum and a mock observed profile when real data are absent. The all-galaxy loop draws the Sérsic index from `N(1.0, 0.5)`. `fit_one_galaxy` itself defaulted to `N(4.0, 0.5)`. Both priors are arguments below.

`USE_PANDEIA = False` uses an analytic Sérsic profile convolved in quadrature with a Gaussian PSF. Set it `True` once the setup cell finds `pandeia_data-2026.2-jwst` and `pandeia_psfs-2026.2-jwst` at the repo root (`pandeia.engine` 2026.2). Optional: also set `PYSYN_CDBS` if you need Synphot atlases.

With real Pandeia, 75 Monte Carlo steps per galaxy is slow (the original notebook estimates ~30 s per `perform_calculation`).


In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize  # Nelder-Mead fit of size and Sérsic index

# False: skip the JWST instrument simulator and use a simple blurred Sérsic profile instead.
# True: call Pandeia, which needs reference data installed on disk.
USE_PANDEIA = True
FILTER = "f200w"  # NIRCam filter centered near 2 microns
# NIRCam plate scale. sw = short-wavelength detector, lw = long-wavelength detector.
PIXEL_SCALE = {"sw": 0.031, "lw": 0.063}  # arcsec / pixel
# How wide the telescope blur is in each filter, as a full width at half maximum, in arcsec.
PSF_FWHM = {
    "f090w": 0.034, "f115w": 0.040, "f150w": 0.050, "f200w": 0.066,
    "f277w": 0.091, "f356w": 0.116, "f444w": 0.145,
}
# Filters that land on the long-wavelength detector, so they use the coarser pixel scale.
LW_FILTERS = {
    "f250m", "f277w", "f300m", "f322w2", "f335m", "f356w", "f360m",
    "f410m", "f430m", "f444w", "f460m", "f480m",
}
RUN_FITS = True  # the example fit at the bottom stays off until you turn this on

if USE_PANDEIA:
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if not os.path.isdir(os.path.join(_repo_root, "pandeia_data-2026.2-jwst")):
        _repo_root = os.getcwd()  # already sitting at the repo root
    _refdata = os.path.join(_repo_root, "pandeia_data-2026.2-jwst")
    _psf_dir = os.path.join(_repo_root, "pandeia_psfs-2026.2-jwst")
    if not os.path.isfile(os.path.join(_refdata, "VERSION_DATA")):
        raise RuntimeError(f"Missing Pandeia refdata at {_refdata}")
    if not os.path.isfile(os.path.join(_psf_dir, "VERSION_PSF")):
        raise RuntimeError(f"Missing Pandeia PSFs at {_psf_dir}")
    os.environ["pandeia_refdata"] = _refdata
    os.environ["PSF_DIR"] = _psf_dir
    from pandeia.engine.calc_utils import build_default_calc
    from pandeia.engine.perform_calculation import perform_calculation


## 1. Blue-monster catalog

Properties from Ferrara, Pallottini & Sommovigo (2024) Table 1 as transcribed in the draft, plus discovery-paper redshifts. `dA_kpc` is a precomputed angular-diameter distance in kpc (Planck18), used only for pc ↔ arcsec.


In [2]:
# One row per galaxy from Ferrara, Pallottini & Sommovigo (2024) Table 1, plus the discovery redshift.
# z: redshift. AV: dust extinction. re_pc: half-light radius in parsecs.
# logM: log10 of stellar mass in solar masses.
# logxi and logZ: the table's ionization and metallicity columns. The size fit uses z and re_pc.
CATALOG = {
    "GS-z14-0":     dict(z=14.18, AV=0.13, re_pc=260, logM=8.84, logxi=-4.76, logZ=-0.75, field="JADES-GS"),
    "GS-z14-1":     dict(z=13.90, AV=0.20, re_pc=160, logM=8.00, logxi=-4.15, logZ=-1.10, field="JADES-GS"),
    "GS-z13-0":     dict(z=13.20, AV=0.05, re_pc=52,  logM=7.95, logxi=-5.68, logZ=-1.69, field="JADES-GS"),
    "UNCOVER-z13":  dict(z=13.08, AV=0.04, re_pc=309, logM=8.13, logxi=-4.41, logZ=-1.57, field="UNCOVER"),
    "GS-z12-0":     dict(z=12.63, AV=0.05, re_pc=144, logM=7.64, logxi=-4.49, logZ=-1.44, field="JADES-GS"),
    "GHZ2":         dict(z=12.34, AV=0.04, re_pc=105, logM=9.05, logxi=-6.27, logZ=-1.40, field="GLASS"),
    "UNCOVER-z12":  dict(z=12.39, AV=0.19, re_pc=426, logM=8.35, logxi=-3.67, logZ=-1.34, field="UNCOVER"),
    "GS-z11-0":     dict(z=11.58, AV=0.18, re_pc=77,  logM=8.67, logxi=-5.50, logZ=-1.87, field="JADES-GS"),
    "Maisie":       dict(z=11.44, AV=0.07, re_pc=340, logM=8.40, logxi=-4.35, logZ=-1.50, field="CEERS"),
    "CEERS2-588":   dict(z=11.04, AV=0.10, re_pc=477, logM=8.99, logxi=-4.49, logZ=-0.84, field="CEERS"),
    "GN-z11":       dict(z=10.60, AV=0.17, re_pc=64,  logM=8.73, logxi=-5.75, logZ=-0.92, field="JADES-GN"),
    "GS-z10-0":     dict(z=10.38, AV=0.05, re_pc=62,  logM=7.58, logxi=-5.16, logZ=-1.91, field="JADES-GS"),
    "UNCOVER-37126": dict(z=10.25, AV=0.18, re_pc=426, logM=8.16, logxi=-3.51, logZ=-1.50, field="UNCOVER"),
    "MACS0647-JD":  dict(z=10.17, AV=0.01, re_pc=70,  logM=7.50, logxi=-5.67, logZ=-0.90, field="MACS0647"),
    "CEERS2-7929":  dict(z=10.10, AV=0.14, re_pc=520, logM=8.50, logxi=-3.78, logZ=-1.50, field="CEERS"),
}

# Angular-diameter distance in kpc at each catalog redshift (Planck 2018).
# This converts a physical size in parsecs into an angle on the sky, and back.
# 1006e3 kpc is 1006 Mpc. The fit uses this only for that conversion.
DA_KPC = {
    10.10: 1006e3, 10.17: 1002e3, 10.25: 1000e3, 10.38: 997e3,
    10.60: 991e3, 11.04: 979e3, 11.44: 969e3, 11.58: 966e3,
    12.34: 949e3, 12.39: 948e3, 12.63: 943e3, 13.08: 933e3,
    13.20: 931e3, 13.90: 918e3, 14.18: 913e3,
}


def pc_to_arcsec(r_pc, z):
    # Small-angle formula: physical size = angle_in_radians * distance.
    # pi / (180 * 3600) is one arcsec written in radians.
    # If this redshift is missing from the table, fall back to 950 Mpc.
    kpc_per_arcsec = DA_KPC.get(z, 950e3) * np.pi / (180 * 3600)
    return (r_pc / 1000.0) / kpc_per_arcsec  # parsecs -> kpc, then divide by kpc per arcsec


def arcsec_to_pc(theta, z):
    # Inverse of pc_to_arcsec.
    kpc_per_arcsec = DA_KPC.get(z, 950e3) * np.pi / (180 * 3600)
    return theta * kpc_per_arcsec * 1000  # arcsec -> kpc -> parsecs


## 2. Candidate SEDs used in the JADES morphology runs

These are the fixed TLUSTY files, redshifts, radii, and magnifications from the morphology draft. Wavelengths are observed-frame microns and the flux is mJy (`1e26` in the shift, rather than the `1e32` nJy factor in the spectral notebook). Flux shortward of Lyα is zero. Build the actual arrays with `01_spectral_fitting.ipynb` (`redshift_tlusty`) or load a two-column file via `load_sed`.

Exposure settings match that draft: NIRCam SW, F200W, `DEEP8`.


In [3]:
# Fixed TLUSTY file, redshift, radius, and flux scale for each JADES morphology run.
# ngroup, nint, nexp: detector reads per ramp, ramps per exposure, and number of exposures.
# theta0: starting angular half-light radius for Nelder-Mead, in arcsec.
# err_anchors rebuild the hand-digitized F200W error bars. The first y0_fixed
# points use a constant bar; the rest are y - lower_tail and upper_tail - y.
JADES_MORPH = {
    "JZ11": dict(
        z=11.37, template="T3M6_0e5.13", R_cm=8.35e14, mu=1.0,
        ngroup=5, nint=1, nexp=35, theta0=0.10,
        profile_csv="JZ11_rad_prof_data.csv",
        err_anchors=dict(
            y0_fixed=3,
            lower_tail=[0.124, 0.067, 0.042, 0.0172, 0.0011],
            upper_tail=[0.154, 0.093, 0.0664, 0.0388, 0.0195],
        ),
    ),
    "JZ13": dict(
        z=13.17, template="T3M4_0e5.13", R_cm=7.632e14, mu=1.0,
        ngroup=5, nint=1, nexp=35, theta0=0.017,  # this object started at 0.017 arcsec
        profile_csv="JZ13_rad_prof_data.csv",
        err_anchors=dict(
            y0_fixed=2,
            lower_tail=[0.18, 0.086, 0.038, 0.0067, 0.0011, 0.0018],
            upper_tail=[0.22, 0.12, 0.068, 0.034, 0.024, 0.026],
        ),
    ),
    "JZ14_0": dict(
        z=14.44, template="ACM2_00e+06.13", R_cm=1.01e15, mu=1.2,  # flux scaled up by 1.2
        ngroup=5, nint=1, nexp=35, theta0=0.02,
        profile_csv="JZ14_0_rad_prof_data.csv",
        err_anchors=dict(
            y0_fixed=6,
            lower_tail=[0.097, 0.076, 0.0855],
            upper_tail=[0.116, 0.096, 0.108],
            floor=0.006,  # constant bar on the inner points; other objects use 0.008
        ),
    ),
    "Maisies": dict(
        z=11.4, template="T3M9_0e5.13", R_cm=8.935e14, mu=1.0,
        ngroup=7, nint=1, nexp=19, theta0=0.05,
        profile_csv=None,
        err_anchors=None,  # morphology draft left this unfinished
    ),
}


def load_sed(path):
    """Two columns: observed wavelength [um], flux [mJy]."""
    data = np.loadtxt(path)
    return data[:, 0], data[:, 1]


def mock_spectrum(z, Muv=-20.0, beta=-2.3):
    """UV power law plus a Lyman break. Placeholder until a joint-fit SED is saved.

    beta is the slope: flux per wavelength scales as lambda**beta. -2.3 is a blue spectrum.
    Muv is the absolute AB magnitude at 1500 angstroms. Brighter galaxies have more negative Muv.
    """
    from astropy.cosmology import Planck18

    wave_rest = np.linspace(800, 5000, 500)  # rest-frame wavelength, angstroms
    f_rest = (wave_rest / 1500.0) ** beta  # power law, normalized to 1 at 1500 A
    f_rest[wave_rest < 912] *= 0.001  # almost no light shortward of the Lyman limit (912 A)
    # Smooth step around Lyman-alpha (1216 A). The 20 A sets how sharp the break is.
    f_rest *= 1.0 / (1.0 + np.exp(-(wave_rest - 1216.0) / 20.0))
    dl_cm = Planck18.luminosity_distance(z).to("cm").value
    # 51.6 converts an AB absolute magnitude into luminosity in erg/s/Hz.
    Lnu_1500 = 10 ** (-0.4 * (Muv - 51.6))
    # Flux at Earth. The extra (1+z) accounts for frequency being stretched by the expansion.
    fnu_1500 = Lnu_1500 / (4 * np.pi * dl_cm ** 2 * (1 + z))
    flam_1500 = fnu_1500 * 3e10 / (1500e-8) ** 2  # f_lambda = f_nu * c / lambda^2 at 1500 A
    f_rest *= flam_1500 / np.interp(1500, wave_rest, f_rest)  # scale the shape to that flux
    wave_um = wave_rest * (1 + z) / 1e4  # rest angstroms -> observed microns
    flam_obs = f_rest / (1 + z)  # stretching the spectrum spreads the same energy over more wavelength
    fnu = flam_obs * (wave_rest * (1 + z) * 1e-8) ** 2 / 3e10  # back to flux per frequency
    flux_mJy = fnu * 1e26  # erg/s/cm^2/Hz -> millijansky. The spectral notebook used 1e32 to get nJy.
    mask = (wave_um > 0.6) & (wave_um < 5.3)  # NIRCam wavelength window
    return wave_um[mask], flux_mJy[mask]


## 3. Image and radial profile

Pandeia scene: input SED in fnu, circular Sérsic (`major = minor = θ`), normalization `none` so the input spectrum sets the flux. The SNR map is azimuthally averaged so background does not dominate the profile.

Mock mode uses the 1D Sérsic law with `b_n ≈ 1.9992 n − 0.3271` and `r_eff` broadened by the Gaussian PSF sigma.


In [4]:
def sersic_1d(r, I_e, r_e, n):
    # Surface brightness of a Sérsic profile. r_e is the half-light radius.
    # n = 1 is an exponential disk. n = 4 is a steep de Vaucouleurs bulge.
    # b_n is chosen so that half the light falls inside r_e.
    b_n = 1.9992 * n - 0.3271
    return I_e * np.exp(-b_n * ((r / r_e) ** (1.0 / n) - 1.0))


def build_pandeia_calc(wave_um, flux_mJy, theta_arcsec, n_sersic, *,
                       nircam_filter="f200w", ngroup=10, nint=1, nexp=1,
                       readout_pattern="deep8"):
    # Start from Pandeia's default JWST NIRCam short-wavelength imaging setup.
    calc = build_default_calc("jwst", "nircam", "sw_imaging")
    src = calc["scene"][0]
    # Feed in the spectrum as flux per frequency. Do not let Pandeia rescale it.
    src["spectrum"]["sed"] = {
        "sed_type": "input",
        "spectrum": [np.asarray(wave_um).tolist(), np.asarray(flux_mJy).tolist()],
        "unit": "fnu",
    }
    src["spectrum"]["normalization"] = {"type": "none"}
    # Circular galaxy: major and minor axes are both the half-light radius, in arcsec.
    # integ_infinity means the profile is normalized by its integral out to infinity.
    src["shape"] = {
        "geometry": "sersic",
        "major": float(theta_arcsec),
        "minor": float(theta_arcsec),
        "sersic_index": float(n_sersic),
        "position_angle": 0.0,
        "norm_method": "integ_infinity",
        "surf_area_units": "arcsec^2",
    }
    calc["configuration"]["instrument"]["filter"] = nircam_filter
    # ngroup: reads up one ramp. nint: ramps. nexp: separate exposures. deep8 is a slow, low-noise readout.
    det = calc["configuration"]["detector"]
    det["ngroup"] = ngroup
    det["nint"] = nint
    det["nexp"] = nexp
    det["readout_pattern"] = readout_pattern
    calc["strategy"]["background_subtraction"] = True
    return calc


def azimuthal_average(image, pixel_scale, annulus_width=0.03, max_radius=0.5):
    # Average the image in rings around the center. Width and radius are in arcsec.
    ny, nx = image.shape
    cy, cx = ny / 2.0, nx / 2.0
    y, x = np.mgrid[0:ny, 0:nx]
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2) * pixel_scale
    edges = np.arange(0, int(max_radius / annulus_width) + 1) * annulus_width
    centers, prof, err = [], [], []
    for i in range(len(edges) - 1):
        pix = image[(r >= edges[i]) & (r < edges[i + 1])]
        pix = pix[np.isfinite(pix)]
        centers.append(0.5 * (edges[i] + edges[i + 1]))  # radius at the middle of the ring
        if len(pix):
            prof.append(np.mean(pix))
            err.append(np.std(pix) / np.sqrt(len(pix)))  # uncertainty on the mean
        else:
            prof.append(0.0)
            err.append(0.0)
    return np.array(centers), np.array(prof), np.array(err)


def radial_profile_at(image, center, radii_arcsec, pixel_scale):
    """Mean in bins centered on the observed radii, as in the JADES draft."""
    # Sample the model at the same radii as the data, so the two profiles can be subtracted point by point.
    x, y = np.indices(image.shape)
    r_pix = np.sqrt((x - center[0]) ** 2 + (y - center[1]) ** 2)
    d_pix = np.asarray(radii_arcsec) / pixel_scale
    out = np.zeros(len(d_pix))
    for i, d in enumerate(d_pix):
        if i == 0:
            # Innermost bin: from the center out to halfway toward the next radius.
            half = (d_pix[i + 1] - d) / 2 if len(d_pix) > 1 else d
            mask = r_pix < d + half
        elif i == len(d_pix) - 1:
            # Outermost bin: same width as the gap to the previous point, centered on this radius.
            lo = 0.5 * (d_pix[i - 1] + d)
            hi = d + 0.5 * (d - d_pix[i - 1])
            mask = (r_pix >= lo) & (r_pix < hi)
        else:
            # Middle bins run from the midpoint with the previous radius to the midpoint with the next.
            lo = 0.5 * (d_pix[i - 1] + d)
            hi = 0.5 * (d + d_pix[i + 1])
            mask = (r_pix >= lo) & (r_pix < hi)
        out[i] = image[mask].mean() if np.any(mask) else 0.0
    return out


def simulate_profile(wave_um, flux_mJy, theta, n, radii_arcsec=None, *,
                     nircam_filter="f200w", annulus_width=0.03, max_radius=0.5,
                     ngroup=10, nint=1, nexp=1):
    if USE_PANDEIA:
        calc = build_pandeia_calc(
            wave_um, flux_mJy, theta, n, nircam_filter=nircam_filter,
            ngroup=ngroup, nint=nint, nexp=nexp,
        )
        result = perform_calculation(calc)
        # Signal-to-noise image, so a flat sky background does not dominate the shape.
        image = result["2d"]["snr"]
        pix = PIXEL_SCALE["lw"] if nircam_filter.lower() in LW_FILTERS else PIXEL_SCALE["sw"]
        if radii_arcsec is None:
            return azimuthal_average(image, pix, annulus_width, max_radius)
        center = np.unravel_index(np.nanargmax(image), image.shape)  # brightest pixel is the center
        prof = radial_profile_at(image, center, radii_arcsec, pix)
        return np.asarray(radii_arcsec), prof, np.zeros_like(prof)

    # Mock path: no instrument simulator. Blur the galaxy by adding its size and the PSF size in quadrature.
    # That shortcut is exact for Gaussians. A real convolution would blur the image with the PSF.
    fwhm = PSF_FWHM.get(nircam_filter.lower(), 0.066)
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))  # Gaussian sigma from the full width at half maximum
    r_eff = np.sqrt(theta ** 2 + sigma ** 2)
    if radii_arcsec is None:
        r = np.arange(annulus_width / 2, max_radius, annulus_width)
    else:
        r = np.asarray(radii_arcsec, float)
    # The 1e-4 arcsec floor keeps r = 0 from blowing up when n is small.
    prof = sersic_1d(np.maximum(r, 1e-4), 1.0, max(r_eff, 1e-4), n)
    err = 1e-4 + 0.005 * (r / max_radius)  # placeholder noise that grows mildly with radius
    return r, prof, err


## 4. Observed profiles

Real profiles are headerless CSVs: radius [arcsec], normalized surface brightness. JADES error bars in the draft are not in the file. `errors_from_anchors` rebuilds them from the stored tails: the first `y0_fixed` points use a constant bar (`floor`, default 0.008), and the rest are `y - lower_tail` and `upper_tail - y`.

`mock_observed_profile` builds a noisy Sérsic with a hidden true index in `[0.5, 2]` for a recovery test.


In [5]:
def load_radial_csv(path):
    # Headerless file: column 0 is radius in arcsec, column 1 is normalized surface brightness.
    data = pd.read_csv(path, header=None)
    return data.iloc[:, 0].to_numpy(float), data.iloc[:, 1].to_numpy(float)


def errors_from_anchors(y, anchors, floor=0.008):
    # Rebuild error bars that were read off a plot by hand.
    # The first y0_fixed points get a constant bar. The rest use the stored ends of each bar.
    if anchors is None:
        return np.full_like(y, floor), np.full_like(y, floor)
    n_fixed = anchors["y0_fixed"]
    floor = anchors.get("floor", floor)
    lower = np.empty_like(y)
    upper = np.empty_like(y)
    lower[:n_fixed] = floor
    upper[:n_fixed] = floor
    lower[n_fixed:] = y[n_fixed:] - np.asarray(anchors["lower_tail"])  # distance down to the lower end
    upper[n_fixed:] = np.asarray(anchors["upper_tail"]) - y[n_fixed:]  # distance up to the upper end
    return lower, upper


def sampler_sigma(lower_err):
    """Draft Monte Carlo width: sigma = 2 * lower_err / 3.92.

    The morphology notebook set upper = y + lower_err and lower = y - lower_err,
    then took (upper - lower) / 3.92. A Gaussian 95% interval is about ±1.96 sigma,
    so the full width is 3.92 sigma. This treats the quoted bar as a 95% half-width.
    """
    return 2.0 * np.asarray(lower_err, float) / 3.92


def mock_observed_profile(name, info, annulus_width=0.03, max_radius=0.4):
    # Same galaxy name always makes the same fake data.
    rng = np.random.default_rng(abs(hash(name)) % (2 ** 31))
    theta = pc_to_arcsec(info["re_pc"], info["z"])  # catalog size, turned into an angle
    n_true = float(rng.uniform(0.5, 2.0))  # hidden true index, disk-like, for a recovery test
    sigma = PSF_FWHM["f200w"] / (2 * np.sqrt(2 * np.log(2)))
    r = np.arange(annulus_width / 2, max_radius, annulus_width)
    prof = sersic_1d(r, 1.0, np.sqrt(theta ** 2 + sigma ** 2), n_true)
    snr = np.maximum(30.0 * prof / prof.max(), 1.0)  # peak signal-to-noise is 30, and it falls with the profile
    err = prof / snr
    noisy = np.maximum(prof + rng.normal(0, err), 0)  # add noise, then clip negative brightness to zero
    return r, noisy, err, dict(theta_true=theta, n_true=n_true)


## 5. Monte Carlo fit

Each draw perturbs the observed profile and restarts Nelder–Mead. Bounds are `θ ∈ (0.001, 0.5)` arcsec and `n ∈ (0.1, 8)`. Both profiles are peak-normalized before the squared residual, matching both drafts.

`n_prior = (1.0, 0.5)` is the blue-monster loop. `n_prior = (4.0, 0.033)` is the JADES-GS-z13 run.


In [ ]:
def profile_mse(params, wave, flux, r_obs, prof_obs, sim_kwargs):
    theta, n = params
    # Soft rejection: a huge loss if the size or the index leaves the allowed range.
    # theta is in arcsec. n from 0.1 to 8 runs from a very flat profile to a very peaked one.
    if not (0.001 < theta < 0.5 and 0.1 < n < 8):
        return 1e10
    try:
        _, sim, _ = simulate_profile(wave, flux, theta, n, radii_arcsec=r_obs, **sim_kwargs)
    except Exception:
        return 1e10
    if not np.any(np.isfinite(sim)) or np.nanmax(sim) <= 0 or np.nanmax(prof_obs) <= 0:
        return 1e10
    # Divide each profile by its peak, so the fit compares shape and ignores overall brightness.
    sim_n = sim / np.nanmax(sim)
    obs_n = prof_obs / np.nanmax(prof_obs)
    return float(np.mean((obs_n - sim_n) ** 2))


def fit_profile(wave, flux, r_obs, prof_obs, err_obs, *, z, re_pc,
                n_mc=75, n_prior=(1.0, 0.5), theta_init=None, seed=None,
                sim_kwargs=None):
    # n_prior = (mean, width). (1.0, 0.5) starts near a disk. (4.0, 0.033) starts very near a bulge.
    rng = np.random.default_rng(seed)
    sim_kwargs = sim_kwargs or {}
    if theta_init is None:
        theta_init = max(pc_to_arcsec(re_pc, z), 0.005)  # catalog size, but not smaller than 0.005 arcsec
    theta_all, n_all, mse_all = [], [], []
    for _ in range(n_mc):
        sample = np.maximum(rng.normal(prof_obs, err_obs), 0)  # one noisy copy of the observed profile
        n0 = float(np.clip(rng.normal(*n_prior), 0.3, 6.0))  # starting index, kept inside 0.3 to 6
        th0 = theta_init * float(rng.uniform(0.5, 1.5))  # starting size, randomly 50% smaller to 50% larger
        res = minimize(
            profile_mse, [th0, n0],
            args=(wave, flux, r_obs, sample, sim_kwargs),
            method="Nelder-Mead",
            # adaptive=True lets the simplex stretch when angle and index have very different scales.
            options={"maxiter": 300, "xatol": 1e-5, "fatol": 1e-8, "adaptive": True},
        )
        theta_all.append(res.x[0])
        n_all.append(res.x[1])
        mse_all.append(res.fun)
    theta_all = np.array(theta_all)
    n_all = np.array(n_all)
    mse_all = np.array(mse_all)
    best = int(np.argmin(mse_all))  # the single draw with the smallest squared error
    return dict(
        theta=theta_all, n=n_all, mse=mse_all,
        theta_med=float(np.median(theta_all)), theta_std=float(np.std(theta_all)),
        n_med=float(np.median(n_all)), n_std=float(np.std(n_all)),
        theta_best=float(theta_all[best]), n_best=float(n_all[best]),
        re_pc_med=float(arcsec_to_pc(np.median(theta_all), z)),  # median angle turned back into parsecs
    )


: 

## 6. Run

`fit_blue_monsters` uses mock SEDs and mock profiles for every catalog entry. `fit_jades_candidate` expects the radial-profile CSV and, if `USE_PANDEIA` is on, a TLUSTY file resolved by the spectral notebook. Both are gated by `RUN_FITS`.


In [ ]:
def fit_blue_monsters(names=None, n_mc=75, n_prior=(1.0, 0.5)):
    # Mock spectrum and mock profile for every catalog galaxy. No real image is read.
    names = list(names or CATALOG)
    results = {}
    for name in names:
        info = CATALOG[name]
        # More massive galaxies are given a brighter UV magnitude: 0.5 mag per dex of mass.
        wave, flux = mock_spectrum(info["z"], Muv=-20.0 - 0.5 * (info["logM"] - 8.5), beta=-2.3)
        r, prof, err, truth = mock_observed_profile(name, info)
        results[name] = fit_profile(
            wave, flux, r, prof, err, z=info["z"], re_pc=info["re_pc"],
            n_mc=n_mc, n_prior=n_prior, seed=0,
        )
        results[name]["truth"] = truth  # hidden size and index, so a recovery test can be checked
        results[name]["r"] = r
        results[name]["profile"] = prof
        results[name]["err"] = err
        print(f"{name}: theta={results[name]['theta_med']:.4f}±{results[name]['theta_std']:.4f} arcsec  "
              f"n={results[name]['n_med']:.2f}±{results[name]['n_std']:.2f}")
    return results


def fit_jades_candidate(name, wave_um, flux_mJy, n_mc=75, n_prior=(4.0, 0.033)):
    """wave/flux in um and mJy. Profile CSV path is JADES_MORPH[name]['profile_csv'].

    n_prior (4.0, 0.033) starts the Sérsic index very close to 4, as in the JADES-GS-z13 run.
    """
    cfg = JADES_MORPH[name]
    r, prof = load_radial_csv(cfg["profile_csv"])
    lower, upper = errors_from_anchors(prof, cfg["err_anchors"])
    # The draft passed the lower error into the noise model, then converted that bar into a Gaussian sigma.
    err = sampler_sigma(lower)
    return fit_profile(
        wave_um, flux_mJy, r, prof, err,
        z=cfg["z"], re_pc=100,  # unused as a start, because theta_init is set just below
        n_mc=n_mc, n_prior=n_prior, theta_init=cfg["theta0"], seed=0,
        sim_kwargs=dict(ngroup=cfg["ngroup"], nint=cfg["nint"], nexp=cfg["nexp"]),
    )


if RUN_FITS:
    # One galaxy, 20 noisy restarts, starting near a disk (n = 1).
    one = fit_blue_monsters(["GS-z14-0"], n_mc=20, n_prior=(1.0, 0.5))
